# Dataset Expansion and Analysis

## Objective

This notebook analyses the current human-annotated literary dataset and prepares strategies for expanding the dataset for advanced machine learning models.

The objectives are:

1. Understand the current annotation distribution.
2. Analyse class balance.
3. Examine text characteristics.
4. Prepare a scalable annotation workflow.
5. Identify requirements for building a larger research dataset.

## Loading Annotated Dataset

The dataset contains literary passages from Frankenstein with human-generated annotations representing therapeutic processes.

In [1]:
import pandas as pd

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
from google.colab import files

uploaded = files.upload()

Saving frankenstein_annotated_pilot.csv to frankenstein_annotated_pilot.csv


In [3]:
df = pd.read_csv("frankenstein_annotated_pilot.csv")

df.head()

,passage_id,text,therapeutic_label,mechanism,intensity,annotator_notes
0,1,"Frankenstein; or, the Modern Prometheus by Mar...",NaN,NaN,NaN,NaN
1,2,These reflections have dispelled the agitation...,NaN,NaN,NaN,NaN
2,3,Six years have passed since I resolved on my p...,NaN,NaN,NaN,NaN
3,4,This is the most favourable period for travell...,NaN,NaN,NaN,NaN
4,5,But I have one want which I have never yet bee...,NaN,NaN,NaN,NaN


In [4]:
print("Total passages:", len(df))

print("Columns:")
print(df.columns)

Total passages: 229
Columns:
Index(['passage_id', 'text', 'therapeutic_label', 'mechanism', 'intensity',
       'annotator_notes'],
      dtype='object')


In [5]:
df["therapeutic_label"].value_counts()

,count
therapeutic_label,
0.0,5
1.0,5


## Annotation Coverage Analysis

The current dataset contains literary passages extracted from Frankenstein.

Only a subset of passages has received human annotation.

This analysis measures the current annotation coverage before expanding the dataset.

In [6]:
total_passages = len(df)

annotated_passages = df["therapeutic_label"].notna().sum()

coverage = (annotated_passages / total_passages) * 100

print("Total passages:", total_passages)
print("Annotated passages:", annotated_passages)
print("Annotation coverage:", round(coverage, 2), "%")

Total passages: 229
Annotated passages: 10
Annotation coverage: 4.37 %


## AI-Assisted Annotation Prioritization

To expand the dataset efficiently, passages will be prioritized based on linguistic indicators related to emotional processing, reflection, memory, and transformation.

The ranking system assists human annotation but does not replace literary interpretation.

In [7]:
therapeutic_keywords = [
    "memory",
    "remember",
    "reflection",
    "suffer",
    "sorrow",
    "hope",
    "despair",
    "fear",
    "grief",
    "heart",
    "past",
    "understand",
    "forgive",
    "love"
]


def keyword_score(text):
    text = text.lower()
    score = 0

    for word in therapeutic_keywords:
        if word in text:
            score += 1

    return score

In [8]:
df["keyword_score"] = df["text"].apply(keyword_score)

df[["passage_id","keyword_score","text"]].sort_values(
    by="keyword_score",
    ascending=False
).head(10)

,passage_id,keyword_score,text
79,80,9,Chapter 9 Nothing is more painful to the human...
76,77,9,"She paused, weeping, and then continued, “I th..."
78,79,7,And on the morrow Justine died. Elizabeth’s he...
194,195,7,I took the hand of Elizabeth. “You are sorrowf...
124,125,7,"“Several changes, in the meantime, took place ..."
14,15,6,"Such words, you may imagine, strongly excited ..."
81,82,5,At these moments I wept bitterly and wished th...
80,81,5,"This advice, although good, was totally inappl..."
77,78,5,"I could not answer. “No, Justine,” said Elizab..."
52,53,5,"“Justine, you may remember, was a great favour..."


## Creating Unannotated Candidate Pool

Previously annotated passages are removed from the prioritization list.

The remaining passages will form the candidate pool for future human annotation.

In [9]:
unannotated_df = df[df["therapeutic_label"].isna()]

print("Remaining passages for annotation:", len(unannotated_df))

Remaining passages for annotation: 219


In [10]:
candidate_queue = (
    unannotated_df[["passage_id","keyword_score","text"]]
    .sort_values(
        by="keyword_score",
        ascending=False
    )
)

candidate_queue.head(20)

,passage_id,keyword_score,text
79,80,9,Chapter 9 Nothing is more painful to the human...
76,77,9,"She paused, weeping, and then continued, “I th..."
78,79,7,And on the morrow Justine died. Elizabeth’s he...
124,125,7,"“Several changes, in the meantime, took place ..."
14,15,6,"Such words, you may imagine, strongly excited ..."
52,53,5,"“Justine, you may remember, was a great favour..."
83,84,5,I listened to this discourse with the extremes...
30,31,5,"She died calmly, and her countenance expressed..."
81,82,5,At these moments I wept bitterly and wished th...
80,81,5,"This advice, although good, was totally inappl..."


## Creating Annotation Batch

The highest-priority unannotated passages are selected for manual review.

Each selected passage will receive human annotations using the established annotation framework.

In [11]:
annotation_batch_01 = candidate_queue.head(20)

annotation_batch_01.to_csv(
    "annotation_batch_01.csv",
    index=False
)

print("Annotation batch created successfully")

Annotation batch created successfully


## Preparing Annotation Fields

Each candidate passage receives structured annotation fields based on the established annotation guidelines.

The fields capture both computational classification and literary interpretation.

In [12]:
batch_df = pd.read_csv("annotation_batch_01.csv")

batch_df.head()

,passage_id,keyword_score,text
0,80,9,Chapter 9 Nothing is more painful to the human...
1,77,9,"She paused, weeping, and then continued, “I th..."
2,79,7,And on the morrow Justine died. Elizabeth’s he...
3,125,7,"“Several changes, in the meantime, took place ..."
4,15,6,"Such words, you may imagine, strongly excited ..."


In [13]:
batch_df["therapeutic_label"] = None
batch_df["mechanism"] = None
batch_df["intensity"] = None
batch_df["annotator_notes"] = None

batch_df.head()

,passage_id,keyword_score,text,therapeutic_label,mechanism,intensity,annotator_notes
0,80,9,Chapter 9 Nothing is more painful to the human...,None,None,None,None
1,77,9,"She paused, weeping, and then continued, “I th...",None,None,None,None
2,79,7,And on the morrow Justine died. Elizabeth’s he...,None,None,None,None
3,125,7,"“Several changes, in the meantime, took place ...",None,None,None,None
4,15,6,"Such words, you may imagine, strongly excited ...",None,None,None,None


In [14]:
batch_df.to_csv(
    "annotation_batch_01_ready.csv",
    index=False
)

print("Annotation template created successfully")

Annotation template created successfully


In [15]:
batch_df[batch_df["passage_id"] == 80]["text"].values[0]

'Chapter 9 Nothing is more painful to the human mind than, after the feelings have\nbeen worked up by a quick succession of events, the dead calmness of\ninaction and certainty which follows and deprives the soul both of hope\nand fear. Justine died, she rested, and I was alive. The blood flowed\nfreely in my veins, but a weight of despair and remorse pressed on my\nheart which nothing could remove. Sleep fled from my eyes; I wandered\nlike an evil spirit, for I had committed deeds of mischief beyond\ndescription horrible, and more, much more (I persuaded myself) was yet\nbehind. Yet my heart overflowed with kindness and the love of virtue.\nI had begun life with benevolent intentions and thirsted for the moment\nwhen I should put them in practice and make myself useful to my fellow\nbeings. Now all was blasted; instead of that serenity of conscience\nwhich allowed me to look back upon the past with self-satisfaction, and\nfrom thence to gather promise of new hopes, I was seized by rem

In [16]:
batch_df.loc[batch_df["passage_id"] == 80, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 80, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 80, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 80, "annotator_notes"] = "The passage presents intense grief, remorse, and self-reflection after Justine's death. Although Victor examines his past identity and suffering, the narrative remains focused on guilt and despair rather than emotional processing, healing, or transformative meaning-making."

In [17]:
batch_df[batch_df["passage_id"] == 77]["text"].values[0]

'She paused, weeping, and then continued, “I thought with horror, my\nsweet lady, that you should believe your Justine, whom your blessed\naunt had so highly honoured, and whom you loved, was a creature capable\nof a crime which none but the devil himself could have perpetrated.\nDear William! dearest blessed child! I soon shall see you again in\nheaven, where we shall all be happy; and that consoles me, going as I\nam to suffer ignominy and death.” “Oh, Justine! Forgive me for having for one moment distrusted you.\nWhy did you confess? But do not mourn, dear girl. Do not fear. I\nwill proclaim, I will prove your innocence. I will melt the stony\nhearts of your enemies by my tears and prayers. You shall not die!\nYou, my playfellow, my companion, my sister, perish on the scaffold!\nNo! No! I never could survive so horrible a misfortune.” Justine shook her head mournfully. “I do not fear to die,” she said;\n“that pang is past. God raises my weakness and gives me courage to\nendure the w

In [18]:
batch_df.loc[batch_df["passage_id"] == 77, "therapeutic_label"] = 1

batch_df.loc[batch_df["passage_id"] == 77, "mechanism"] = "Emotional Catharsis"

batch_df.loc[batch_df["passage_id"] == 77, "intensity"] = 4

batch_df.loc[batch_df["passage_id"] == 77, "annotator_notes"] = "The passage represents emotional catharsis through the expression of grief, fear, and acceptance during Justine's final moments. Elizabeth provides emotional support and reassurance, while Justine demonstrates a movement toward courage and acceptance despite suffering."

In [19]:
batch_df[batch_df["passage_id"] == 79]["text"].values[0]

'And on the morrow Justine died. Elizabeth’s heart-rending eloquence\nfailed to move the judges from their settled conviction in the\ncriminality of the saintly sufferer. My passionate and indignant\nappeals were lost upon them. And when I received their cold answers\nand heard the harsh, unfeeling reasoning of these men, my purposed\navowal died away on my lips. Thus I might proclaim myself a madman,\nbut not revoke the sentence passed upon my wretched victim. She\nperished on the scaffold as a murderess! From the tortures of my own heart, I turned to contemplate the deep and\nvoiceless grief of my Elizabeth. This also was my doing! And my\nfather’s woe, and the desolation of that late so smiling home all was\nthe work of my thrice-accursed hands! Ye weep, unhappy ones, but these\nare not your last tears! Again shall you raise the funeral wail, and\nthe sound of your lamentations shall again and again be heard!\nFrankenstein, your son, your kinsman, your early, much-loved friend; he\n

In [20]:
batch_df.loc[batch_df["passage_id"] == 79, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 79, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 79, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 79, "annotator_notes"] = "The passage presents intense grief, guilt, and self-blame following Justine's execution. Although Victor reflects deeply on the suffering of himself and others, the narrative emphasizes despair and responsibility rather than emotional processing, healing, or transformative meaning-making."

In [21]:
batch_df[batch_df["passage_id"] == 125]["text"].values[0]

'“Several changes, in the meantime, took place in the cottage. The\npresence of Safie diffused happiness among its inhabitants, and I also\nfound that a greater degree of plenty reigned there. Felix and Agatha\nspent more time in amusement and conversation, and were assisted in\ntheir labours by servants. They did not appear rich, but they were\ncontented and happy; their feelings were serene and peaceful, while\nmine became every day more tumultuous. Increase of knowledge only\ndiscovered to me more clearly what a wretched outcast I was. I\ncherished hope, it is true, but it vanished when I beheld my person\nreflected in water or my shadow in the moonshine, even as that frail\nimage and that inconstant shade. “I endeavoured to crush these fears and to fortify myself for the trial\nwhich in a few months I resolved to undergo; and sometimes I allowed my\nthoughts, unchecked by reason, to ramble in the fields of Paradise, and\ndared to fancy amiable and lovely creatures sympathising with

In [22]:
batch_df.loc[batch_df["passage_id"] == 125, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 125, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 125, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 125, "annotator_notes"] = "The passage presents deep self-reflection and emotional awareness as the Creature compares human happiness with his own isolation. However, this reflection leads to increased despair and identity conflict rather than healing, emotional processing, or transformative growth."

In [23]:
batch_df[batch_df["passage_id"] == 15]["text"].values[0]

'Such words, you may imagine, strongly excited my curiosity; but the\nparoxysm of grief that had seized the stranger overcame his weakened\npowers, and many hours of repose and tranquil conversation were\nnecessary to restore his composure. Having conquered the violence of his feelings, he appeared to despise\nhimself for being the slave of passion; and quelling the dark tyranny of\ndespair, he led me again to converse concerning myself personally. He asked\nme the history of my earlier years. The tale was quickly told, but it\nawakened various trains of reflection. I spoke of my desire of finding a\nfriend, of my thirst for a more intimate sympathy with a fellow mind than\nhad ever fallen to my lot, and expressed my conviction that a man could\nboast of little happiness who did not enjoy this blessing. “I agree with you,” replied the stranger; “we are\nunfashioned creatures, but half made up, if one wiser, better, dearer than\nourselves—such a friend ought to be—do not lend his aid to

In [24]:
batch_df.loc[batch_df["passage_id"] == 15, "therapeutic_label"] = 1

batch_df.loc[batch_df["passage_id"] == 15, "mechanism"] = "Narrative Reconstruction"

batch_df.loc[batch_df["passage_id"] == 15, "intensity"] = 4

batch_df.loc[batch_df["passage_id"] == 15, "annotator_notes"] = "The passage represents therapeutic processing through emotional expression, interpersonal conversation, and narrative reconstruction. The stranger moves from intense grief toward composure while sharing personal history and reflecting on the importance of human connection and friendship."

In [25]:
batch_df[batch_df["passage_id"] == 53]["text"].values[0]


'“Justine, you may remember, was a great favourite of yours; and I\nrecollect you once remarked that if you were in an ill humour, one\nglance from Justine could dissipate it, for the same reason that\nAriosto gives concerning the beauty of Angelica—she looked so\nfrank-hearted and happy. My aunt conceived a great attachment for her,\nby which she was induced to give her an education superior to that\nwhich she had at first intended. This benefit was fully repaid;\nJustine was the most grateful little creature in the world: I do not\nmean that she made any professions I never heard one pass her lips, but\nyou could see by her eyes that she almost adored her protectress.\nAlthough her disposition was gay and in many respects inconsiderate,\nyet she paid the greatest attention to every gesture of my aunt. She\nthought her the model of all excellence and endeavoured to imitate her\nphraseology and manners, so that even now she often reminds me of her. “When my dearest aunt died every one 

In [26]:
batch_df.loc[batch_df["passage_id"] == 53, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 53, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 53, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 53, "annotator_notes"] = "The passage presents a recollection of Justine's character and her relationship with the Frankenstein family. Although it contains affection and emotional memory, it functions primarily as descriptive remembrance rather than emotional processing, healing, or transformative reflection."

In [27]:
batch_df[batch_df["passage_id"] == 84]["text"].values[0]

'I listened to this discourse with the extremest agony. I, not in deed,\nbut in effect, was the true murderer. Elizabeth read my anguish in my\ncountenance, and kindly taking my hand, said, “My dearest friend, you\nmust calm yourself. These events have affected me, God knows how\ndeeply; but I am not so wretched as you are. There is an expression of\ndespair, and sometimes of revenge, in your countenance that makes me\ntremble. Dear Victor, banish these dark passions. Remember the\nfriends around you, who centre all their hopes in you. Have we lost\nthe power of rendering you happy? Ah! While we love, while we are\ntrue to each other, here in this land of peace and beauty, your native\ncountry, we may reap every tranquil blessing—what can disturb our\npeace?” And could not such words from her whom I fondly prized before every\nother gift of fortune suffice to chase away the fiend that lurked in my\nheart? Even as she spoke I drew near to her, as if in terror, lest at\nthat very moment 

In [28]:
batch_df.loc[batch_df["passage_id"] == 84, "therapeutic_label"] = 1

batch_df.loc[batch_df["passage_id"] == 84, "mechanism"] = "Emotional Catharsis"

batch_df.loc[batch_df["passage_id"] == 84, "intensity"] = 3

batch_df.loc[batch_df["passage_id"] == 84, "annotator_notes"] = "The passage represents emotional catharsis through Victor's expression of guilt and anguish, followed by Elizabeth's compassionate support. The interaction provides emotional regulation and attempts to redirect Victor from despair toward connection and hope."

In [29]:
batch_df[batch_df["passage_id"] == 31]["text"].values[0]

'She died calmly, and her countenance expressed affection even in death.\nI need not describe the feelings of those whose dearest ties are rent\nby that most irreparable evil, the void that presents itself to the\nsoul, and the despair that is exhibited on the countenance. It is so\nlong before the mind can persuade itself that she whom we saw every day\nand whose very existence appeared a part of our own can have departed\nfor ever—that the brightness of a beloved eye can have been\nextinguished and the sound of a voice so familiar and dear to the ear\ncan be hushed, never more to be heard. These are the reflections of\nthe first days; but when the lapse of time proves the reality of the\nevil, then the actual bitterness of grief commences. Yet from whom has\nnot that rude hand rent away some dear connection? And why should I\ndescribe a sorrow which all have felt, and must feel? The time at\nlength arrives when grief is rather an indulgence than a necessity; and\nthe smile that plays

In [30]:
batch_df.loc[batch_df["passage_id"] == 31, "therapeutic_label"] = 1

batch_df.loc[batch_df["passage_id"] == 31, "mechanism"] = "Narrative Reconstruction"

batch_df.loc[batch_df["passage_id"] == 31, "intensity"] = 3

batch_df.loc[batch_df["passage_id"] == 31, "annotator_notes"] = "The passage represents narrative reconstruction of grief through reflection on loss, memory, and the gradual transformation of sorrow over time. The narrator moves from immediate suffering toward a broader understanding of grief as a universal human experience."

In [31]:
batch_df[batch_df["passage_id"] == 82]["text"].values[0]

'At these moments I wept bitterly and wished that peace would revisit my\nmind only that I might afford them consolation and happiness. But that\ncould not be. Remorse extinguished every hope. I had been the author of\nunalterable evils, and I lived in daily fear lest the monster whom I had\ncreated should perpetrate some new wickedness. I had an obscure feeling\nthat all was not over and that he would still commit some signal crime,\nwhich by its enormity should almost efface the recollection of the past.\nThere was always scope for fear so long as anything I loved remained\nbehind. My abhorrence of this fiend cannot be conceived. When I thought of\nhim I gnashed my teeth, my eyes became inflamed, and I ardently wished to\nextinguish that life which I had so thoughtlessly bestowed. When I\nreflected on his crimes and malice, my hatred and revenge burst all bounds\nof moderation. I would have made a pilgrimage to the highest peak of the\nAndes, could I, when there, have precipitated hi

In [32]:
batch_df.loc[batch_df["passage_id"] == 82, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 82, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 82, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 82, "annotator_notes"] = "The passage presents intense guilt, remorse, fear, and anger following Victor's actions. Although it contains self-reflection, the emotional movement remains focused on despair and revenge rather than healing, acceptance, or transformative processing."

In [33]:
batch_df[batch_df["passage_id"] == 81]["text"].values[0]

'This advice, although good, was totally inapplicable to my case; I\nshould have been the first to hide my grief and console my friends if\nremorse had not mingled its bitterness, and terror its alarm, with my\nother sensations. Now I could only answer my father with a look of\ndespair and endeavour to hide myself from his view. About this time we retired to our house at Belrive. This change was\nparticularly agreeable to me. The shutting of the gates regularly at\nten o’clock and the impossibility of remaining on the lake after that\nhour had rendered our residence within the walls of Geneva very irksome\nto me. I was now free. Often, after the rest of the family had\nretired for the night, I took the boat and passed many hours upon the\nwater. Sometimes, with my sails set, I was carried by the wind; and\nsometimes, after rowing into the middle of the lake, I left the boat to\npursue its own course and gave way to my own miserable reflections. I\nwas often tempted, when all was at pea

In [34]:
batch_df.loc[batch_df["passage_id"] == 81, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 81, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 81, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 81, "annotator_notes"] = "The passage presents grief, remorse, fear, and isolation following traumatic events. Although Victor engages in self-reflection, the reflection reinforces despair and withdrawal rather than producing emotional processing, healing, or transformative understanding."

In [35]:
batch_df[batch_df["passage_id"] == 78]["text"].values[0]

'I could not answer. “No, Justine,” said Elizabeth; “he is more\nconvinced of your innocence than I was, for even when he heard that you\nhad confessed, he did not credit it.” “I truly thank him. In these last moments I feel the sincerest\ngratitude towards those who think of me with kindness. How sweet is\nthe affection of others to such a wretch as I am! It removes more than\nhalf my misfortune, and I feel as if I could die in peace now that my\ninnocence is acknowledged by you, dear lady, and your cousin.” Thus the poor sufferer tried to comfort others and herself. She indeed\ngained the resignation she desired. But I, the true murderer, felt the\nnever-dying worm alive in my bosom, which allowed of no hope or\nconsolation. Elizabeth also wept and was unhappy, but hers also was\nthe misery of innocence, which, like a cloud that passes over the fair\nmoon, for a while hides but cannot tarnish its brightness. Anguish and\ndespair had penetrated into the core of my heart; I bore a hell

In [36]:
batch_df.loc[batch_df["passage_id"] == 78, "therapeutic_label"] = 1

batch_df.loc[batch_df["passage_id"] == 78, "mechanism"] = "Emotional Catharsis"

batch_df.loc[batch_df["passage_id"] == 78, "intensity"] = 4

batch_df.loc[batch_df["passage_id"] == 78, "annotator_notes"] = "The passage represents emotional catharsis through Justine's expression of gratitude, acceptance, and emotional relief after receiving recognition and kindness from others. Social support reduces her suffering and allows her to approach death with peace and resignation."

In [37]:
batch_df[batch_df["passage_id"] == 167]["text"].values[0]


'“Devil, cease; and do not poison the air with these sounds of malice.\nI have declared my resolution to you, and I am no coward to bend\nbeneath words. Leave me; I am inexorable.” “It is well. I go; but remember, I shall be with you on your\nwedding-night.” I started forward and exclaimed, “Villain! Before you sign my\ndeath-warrant, be sure that you are yourself safe.” I would have seized him, but he eluded me and quitted the house with\nprecipitation. In a few moments I saw him in his boat, which shot\nacross the waters with an arrowy swiftness and was soon lost amidst the\nwaves. All was again silent, but his words rang in my ears. I burned with rage to\npursue the murderer of my peace and precipitate him into the ocean. I\nwalked up and down my room hastily and perturbed, while my imagination\nconjured up a thousand images to torment and sting me. Why had I not\nfollowed him and closed with him in mortal strife? But I had suffered him\nto depart, and he had directed his course tow

In [38]:
batch_df.loc[batch_df["passage_id"] == 167, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 167, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 167, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 167, "annotator_notes"] = "The passage presents intense anger, hatred, and desire for revenge following Victor's confrontation with the Creature. Although it contains emotional expression and reflection, the narrative movement is toward retaliation and psychological distress rather than healing, acceptance, or transformative processing."

In [39]:
batch_df[batch_df["passage_id"] == 141]["text"].values[0]

'Chapter 17 The being finished speaking and fixed his looks upon me in the\nexpectation of a reply. But I was bewildered, perplexed, and unable to\narrange my ideas sufficiently to understand the full extent of his\nproposition. He continued, “You must create a female for me with whom I can live in the\ninterchange of those sympathies necessary for my being. This you alone\ncan do, and I demand it of you as a right which you must not refuse to\nconcede.” The latter part of his tale had kindled anew in me the anger that had\ndied away while he narrated his peaceful life among the cottagers, and\nas he said this I could no longer suppress the rage that burned within\nme. “I do refuse it,” I replied; “and no torture shall ever extort a\nconsent from me. You may render me the most miserable of men, but you\nshall never make me base in my own eyes. Shall I create another like\nyourself, whose joint wickedness might desolate the world. Begone! I\nhave answered you; you may torture me, but I 

In [40]:
batch_df.loc[batch_df["passage_id"] == 141, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 141, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 141, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 141, "annotator_notes"] = "The passage presents an emotional confrontation between Victor and the Creature involving loneliness, desire for sympathy, anger, and rejection. Although the characters express personal needs and emotions, the interaction results in conflict rather than emotional processing, healing, or transformative understanding."

In [41]:
batch_df[batch_df["passage_id"] == 157]["text"].values[0]

'I enjoyed this scene, and yet my enjoyment was embittered both by the\nmemory of the past and the anticipation of the future. I was formed\nfor peaceful happiness. During my youthful days discontent never\nvisited my mind, and if I was ever overcome by _ennui_, the sight of what\nis beautiful in nature or the study of what is excellent and sublime in\nthe productions of man could always interest my heart and communicate\nelasticity to my spirits. But I am a blasted tree; the bolt has\nentered my soul; and I felt then that I should survive to exhibit what\nI shall soon cease to be—a miserable spectacle of wrecked humanity,\npitiable to others and intolerable to myself. We passed a considerable period at Oxford, rambling among its environs\nand endeavouring to identify every spot which might relate to the most\nanimating epoch of English history. Our little voyages of discovery\nwere often prolonged by the successive objects that presented\nthemselves. We visited the tomb of the illustr

In [42]:
batch_df.loc[batch_df["passage_id"] == 157, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 157, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 157, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 157, "annotator_notes"] = "The passage presents deep self-reflection about past happiness, present suffering, and future anxiety. Although Victor reconstructs aspects of his identity, the reflection reinforces despair and a sense of personal destruction rather than healing, acceptance, or transformative growth."

In [43]:
batch_df[batch_df["passage_id"] == 151]["text"].values[0]

'It was in the latter end of September that I again quitted my native\ncountry. My journey had been my own suggestion, and Elizabeth\ntherefore acquiesced, but she was filled with disquiet at the idea of\nmy suffering, away from her, the inroads of misery and grief. It had\nbeen her care which provided me a companion in Clerval—and yet a man\nis blind to a thousand minute circumstances which call forth a woman’s\nsedulous attention. She longed to bid me hasten my return; a thousand\nconflicting emotions rendered her mute as she bade me a tearful, silent\nfarewell. I threw myself into the carriage that was to convey me away, hardly\nknowing whither I was going, and careless of what was passing around.\nI remembered only, and it was with a bitter anguish that I reflected on\nit, to order that my chemical instruments should be packed to go with\nme. Filled with dreary imaginations, I passed through many beautiful\nand majestic scenes, but my eyes were fixed and unobserving. I could\nonly 

In [44]:
batch_df.loc[batch_df["passage_id"] == 151, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 151, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 151, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 151, "annotator_notes"] = "The passage presents Victor's emotional suffering, grief, and withdrawal during his separation from Elizabeth. Although it includes interpersonal care and concern, Victor remains trapped in despair and obsessive thoughts rather than engaging in emotional processing, healing, or transformative reflection."

In [45]:
batch_df[batch_df["passage_id"] == 226]["text"].values[0]

'“Oh, it is not thus—not thus,” interrupted the being.\n“Yet such must be the impression conveyed to you by what appears to\nbe the purport of my actions. Yet I seek not a fellow feeling in my misery.\nNo sympathy may I ever find. When I first sought it, it was the love of\nvirtue, the feelings of happiness and affection with which my whole being\noverflowed, that I wished to be participated. But now that virtue has\nbecome to me a shadow, and that happiness and affection are turned into\nbitter and loathing despair, in what should I seek for sympathy? I am\ncontent to suffer alone while my sufferings shall endure; when I die, I am\nwell satisfied that abhorrence and opprobrium should load my memory. Once\nmy fancy was soothed with dreams of virtue, of fame, and of enjoyment. Once\nI falsely hoped to meet with beings who, pardoning my outward form, would\nlove me for the excellent qualities which I was capable of unfolding. I was\nnourished with high thoughts of honour and devotion. Bu

In [46]:
batch_df.loc[batch_df["passage_id"] == 226, "therapeutic_label"] = 1

batch_df.loc[batch_df["passage_id"] == 226, "mechanism"] = "Narrative Reconstruction"

batch_df.loc[batch_df["passage_id"] == 226, "intensity"] = 4

batch_df.loc[batch_df["passage_id"] == 226, "annotator_notes"] = "The passage represents narrative reconstruction through the Creature's reflection on his past hopes, emotional suffering, and transformation of identity. Although he remains isolated and despairing, the passage demonstrates meaningful self-examination and an attempt to understand his experience."

In [47]:
batch_df[batch_df["passage_id"] == 177]["text"].values[0]

'A fever succeeded to this. I lay for two months on the point of death; my\nravings, as I afterwards heard, were frightful; I called myself the\nmurderer of William, of Justine, and of Clerval. Sometimes I entreated my\nattendants to assist me in the destruction of the fiend by whom I was\ntormented; and at others I felt the fingers of the monster already grasping\nmy neck, and screamed aloud with agony and terror. Fortunately, as I spoke\nmy native language, Mr. Kirwin alone understood me; but my gestures and\nbitter cries were sufficient to affright the other witnesses. Why did I not die? More miserable than man ever was before, why did I not\nsink into forgetfulness and rest? Death snatches away many blooming\nchildren, the only hopes of their doting parents; how many brides and\nyouthful lovers have been one day in the bloom of health and hope, and the\nnext a prey for worms and the decay of the tomb! Of what materials was I\nmade that I could thus resist so many shocks, which, lik

In [48]:
batch_df.loc[batch_df["passage_id"] == 177, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 177, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 177, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 177, "annotator_notes"] = "The passage presents severe psychological suffering, trauma, and guilt following multiple deaths. Although Victor expresses his emotions and recognizes his responsibility, the narrative remains focused on despair, fear, and self-condemnation rather than emotional processing, healing, or transformative understanding."

In [49]:
batch_df[batch_df["passage_id"] == 190]["text"].values[0]

'“Do not let this letter disturb you; do not answer tomorrow, or the\nnext day, or even until you come, if it will give you pain. My uncle\nwill send me news of your health, and if I see but one smile on your\nlips when we meet, occasioned by this or any other exertion of mine, I\nshall need no other happiness. “Elizabeth Lavenza. “Geneva, May 18th, 17—” This letter revived in my memory what I had before forgotten, the threat of\nthe fiend—“_I will be with you on your\nwedding-night!_” Such was my sentence, and on that night would the\ndæmon employ every art to destroy me and tear me from the glimpse of\nhappiness which promised partly to console my sufferings. On that night he\nhad determined to consummate his crimes by my death. Well, be it so; a\ndeadly struggle would then assuredly take place, in which if he were\nvictorious I should be at peace and his power over me be at an end. If he\nwere vanquished, I should be a free man. Alas! What freedom? Such as the\npeasant enjoys when h

In [50]:
batch_df.loc[batch_df["passage_id"] == 190, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 190, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 190, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 190, "annotator_notes"] = "The passage contains Elizabeth's compassionate support and emotional reassurance toward Victor. However, Victor's response is dominated by fear, anxiety, and anticipation of conflict with the Creature. The passage does not develop into emotional processing, healing, or transformative understanding."

In [51]:
batch_df[batch_df["passage_id"] == 211]["text"].values[0]

'On hearing this information I suffered a temporary access of despair.\nHe had escaped me, and I must commence a destructive and almost endless\njourney across the mountainous ices of the ocean, amidst cold that few\nof the inhabitants could long endure and which I, the native of a\ngenial and sunny climate, could not hope to survive. Yet at the idea\nthat the fiend should live and be triumphant, my rage and vengeance\nreturned, and like a mighty tide, overwhelmed every other feeling.\nAfter a slight repose, during which the spirits of the dead hovered\nround and instigated me to toil and revenge, I prepared for my journey. I exchanged my land-sledge for one fashioned for the inequalities of\nthe Frozen Ocean, and purchasing a plentiful stock of provisions, I\ndeparted from land. I cannot guess how many days have passed since then, but I have endured\nmisery which nothing but the eternal sentiment of a just retribution\nburning within my heart could have enabled me to support. Immense 

In [52]:
batch_df.loc[batch_df["passage_id"] == 211, "therapeutic_label"] = 0

batch_df.loc[batch_df["passage_id"] == 211, "mechanism"] = "None"

batch_df.loc[batch_df["passage_id"] == 211, "intensity"] = 0

batch_df.loc[batch_df["passage_id"] == 211, "annotator_notes"] = "The passage presents Victor's despair, rage, and desire for revenge after the Creature's escape. Although it contains emotional expression and reflection on suffering, the narrative movement is toward vengeance and obsession rather than healing, acceptance, or transformative understanding."

In [53]:
batch_df.to_csv(
    "annotation_batch_01_completed.csv",
    index=False
)

print("Batch 01 annotation completed")

Batch 01 annotation completed


In [54]:
batch_df.to_csv(
    "annotation_batch_01_completed.csv",
    index=False
)

print("Batch 01 annotation completed")

Batch 01 annotation completed


## Merging Annotation Batches

The pilot dataset and expanded annotation batches are combined to create a larger supervised learning dataset.

This dataset will be used for future machine learning experiments.

In [55]:
pilot_df = pd.read_csv("frankenstein_annotated_pilot.csv")

batch_completed_df = pd.read_csv(
    "annotation_batch_01_completed.csv"
)

expanded_df = pd.concat(
    [pilot_df, batch_completed_df],
    ignore_index=True
)

print("Total annotated passages:", len(expanded_df))

Total annotated passages: 249


In [56]:
print("Total rows:", len(expanded_df))

print(
    "Actually annotated:",
    expanded_df["therapeutic_label"].notna().sum()
)

Total rows: 249
Actually annotated: 29


In [57]:
expanded_df[
    expanded_df["therapeutic_label"].isna()
].head()

,passage_id,text,therapeutic_label,mechanism,intensity,annotator_notes,keyword_score
0,1,"Frankenstein; or, the Modern Prometheus by Mar...",NaN,NaN,NaN,NaN,NaN
1,2,These reflections have dispelled the agitation...,NaN,NaN,NaN,NaN,NaN
2,3,Six years have passed since I resolved on my p...,NaN,NaN,NaN,NaN,NaN
3,4,This is the most favourable period for travell...,NaN,NaN,NaN,NaN,NaN
4,5,But I have one want which I have never yet bee...,NaN,NaN,NaN,NaN,NaN


In [58]:
expanded_df["therapeutic_label"].value_counts(dropna=False)

,count
therapeutic_label,
NaN,220
0.0,18
1.0,11


## Annotation Mechanism Distribution

This analysis examines the distribution of therapeutic mechanisms identified during human annotation.

The purpose is to understand which therapeutic processes appear most frequently in the literary dataset.

In [59]:
expanded_df["mechanism"].value_counts(dropna=False)

,count
mechanism,
NaN,238
Narrative Reconstruction,6
Emotional Catharsis,5


In [60]:
expanded_df["intensity"].value_counts(dropna=False)

,count
intensity,
NaN,220
0.0,18
3.0,6
4.0,5


In [61]:
expanded_df["mechanism"].value_counts(dropna=False)

,count
mechanism,
NaN,238
Narrative Reconstruction,6
Emotional Catharsis,5


## Creating Clean Annotated Dataset

Only passages with complete human annotations are retained for supervised machine learning experiments.

The resulting dataset contains verified examples with labels, mechanisms, intensity scores, and annotation reasoning.

In [62]:
clean_dataset = expanded_df[
    expanded_df["therapeutic_label"].notna()
]

print("Clean dataset size:", len(clean_dataset))

Clean dataset size: 29


In [63]:
clean_dataset.to_csv(
    "frankenstein_training_dataset.csv",
    index=False
)

print("Clean training dataset saved successfully")

Clean training dataset saved successfully


In [64]:
from google.colab import files

files.download("frankenstein_training_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>